# XGBoost Forecasting — Training & Verification

This notebook walks through the full pipeline:

1. **Data loading** — read live telemetry CSVs
2. **Preprocessing** — resample to 5s buckets
3. **Feature engineering** — build lag feature matrix
4. **Training** — fit XGBRegressor per signal
5. **Model inspection** — feature importances, prediction vs actual
6. **Residual analysis** — z-score distribution, anomaly threshold
7. **Anomaly injection tests** — inject artificial attacks, verify detection
8. **Inference simulation** — step through the live deque-based inference loop
9. **All-signals test** — verify all 7 loaded models produce sensible predictions

Run from `backend/` directory, or adjust `BACKEND_DIR` below.

In [ ]:
import sys
from pathlib import Path

# Ensure backend/ is on sys.path so forecasting.* imports work
BACKEND_DIR = Path("__file__").resolve().parent.parent.parent
# Fallback: if running directly from backend/
if not (BACKEND_DIR / "forecasting").exists():
    BACKEND_DIR = Path(".").resolve()
if not (BACKEND_DIR / "forecasting").exists():
    BACKEND_DIR = Path(".").resolve().parent

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print(f"Backend dir: {BACKEND_DIR}")
print(f"forecasting/ exists: {(BACKEND_DIR / 'forecasting').exists()}")

In [ ]:
from collections import deque

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
from xgboost import XGBRegressor

from forecasting.config import (
    ANOMALY_Z_THRESHOLD,
    LAG_WINDOW,
    LOG_DIR,
    MIN_TRAINING_BUCKETS,
    MODELS_DIR,
    RESIDUAL_STATS_PATH,
    SIGNALS,
    SUSTAINED_ANOMALY_THRESHOLD,
    XGB_LEARNING_RATE,
    XGB_MAX_DEPTH,
    XGB_N_ESTIMATORS,
)
from forecasting.preprocess import load_sensor_logs, resample_to_5s
import forecasting.state as state_mod

plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print(f"LAG_WINDOW        = {LAG_WINDOW}")
print(f"XGB_N_ESTIMATORS  = {XGB_N_ESTIMATORS}")
print(f"XGB_MAX_DEPTH     = {XGB_MAX_DEPTH}")
print(f"XGB_LEARNING_RATE = {XGB_LEARNING_RATE}")
print(f"ANOMALY_Z_THRESH  = {ANOMALY_Z_THRESHOLD}")
print(f"SUSTAINED_THRESH  = {SUSTAINED_ANOMALY_THRESHOLD} consecutive buckets")
print(f"LOG_DIR           = {LOG_DIR}")

---
## 1. Data Loading

Load all CSVs for one signal and inspect the raw data.

In [ ]:
# Pick one signal to demonstrate end-to-end
DEMO_SIGNAL = "load1_ac_voltage"
cfg = SIGNALS[DEMO_SIGNAL]

raw = load_sensor_logs(cfg["sensor"], cfg["field"], LOG_DIR)

print(f"Signal  : {DEMO_SIGNAL}")
print(f"Sensor  : {cfg['sensor']}")
print(f"Field   : {cfg['field']}")
print(f"Rows    : {len(raw):,}")
print(f"Range   : {raw.index[0]}  →  {raw.index[-1]}")
print(f"Mean    : {raw.mean():.3f}")
print(f"Std     : {raw.std():.3f}")
print(f"Min/Max : {raw.min():.3f} / {raw.max():.3f}")
raw.head(10)

In [ ]:
fig, ax = plt.subplots()
raw.plot(ax=ax, linewidth=0.6, alpha=0.8, color="steelblue")
ax.set_title(f"{DEMO_SIGNAL} — raw readings (every 0.5s)")
ax.set_ylabel(cfg["field"])
ax.set_xlabel("timestamp (UTC)")
plt.tight_layout()
plt.show()

---
## 2. Preprocessing — Resample to 5s Buckets

The model trains on 5-second mean buckets, not raw 0.5s readings.
This smooths noise and gives evenly-spaced time steps.

In [ ]:
resampled = resample_to_5s(raw)

print(f"Raw rows      : {len(raw):,}")
print(f"Resampled rows: {len(resampled):,}  ({len(raw)/len(resampled):.1f}x compression)")
print(f"Min buckets needed: {MIN_TRAINING_BUCKETS}")
print(f"We have       : {len(resampled):,}  ({'OK' if len(resampled) >= MIN_TRAINING_BUCKETS else 'NOT ENOUGH — run mock_training_data.py'})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

raw.iloc[:600].plot(ax=axes[0], linewidth=0.5, color="steelblue", alpha=0.7)
axes[0].set_title("Raw readings (first 5 minutes)")
axes[0].set_ylabel("value")

resampled.iloc[:60].plot(ax=axes[1], linewidth=1.2, color="darkorange", marker="o", markersize=3)
axes[1].set_title("Resampled to 5s (first 5 minutes)")
axes[1].set_ylabel("mean value per 5s bucket")

plt.tight_layout()
plt.show()

---
## 3. Feature Engineering — Lag Matrix

Each row of the training matrix represents one 5s bucket.

| Feature | Meaning |
|---------|--------|
| `lag_1` | value from 5s ago |
| `lag_2` | value from 10s ago |
| `lag_3` | value from 15s ago |
| `lag_4` | value from 20s ago |
| `lag_5` | value from 25s ago |
| `hour` | hour of day (0–23) |
| `minute` | minute of hour (0–59) |
| **target** | **current value — what we predict** |

In [ ]:
def build_features(series: pd.Series, lag: int):
    """Build (X, y) arrays — same logic as train.py."""
    vals = series.values
    idx = series.index
    X, y = [], []
    for i in range(lag, len(vals)):
        lags = vals[i - lag:i][::-1]   # [t-1, t-2, ..., t-lag]
        ts = idx[i]
        X.append([*lags, ts.hour, ts.minute])
        y.append(vals[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = build_features(resampled, LAG_WINDOW)

feature_names = [f"lag_{i+1}" for i in range(LAG_WINDOW)] + ["hour", "minute"]

print(f"X shape: {X.shape}  ({X.shape[0]} training samples, {X.shape[1]} features)")
print(f"y shape: {y.shape}")

# Show a sample
df_sample = pd.DataFrame(X[:5], columns=feature_names)
df_sample["target"] = y[:5]
df_sample.round(3)

In [ ]:
# Check: lag_1 for row i should equal target for row i-1
assert abs(X[1, 0] - y[0]) < 1e-5, "lag_1 doesn't match previous target!"
print("Lag consistency check passed: X[1, lag_1] == y[0]")
print(f"  y[0]       = {y[0]:.4f}  (target for bucket 0)")
print(f"  X[1, lag_1]= {X[1, 0]:.4f}  (should be same value, 5s earlier)")

---
## 4. Training

Fit an XGBRegressor on the demo signal. This is the same code as `train.py`.

In [ ]:
model = XGBRegressor(
    n_estimators=XGB_N_ESTIMATORS,
    max_depth=XGB_MAX_DEPTH,
    learning_rate=XGB_LEARNING_RATE,
    tree_method="hist",
    verbosity=0,
)
model.fit(X, y)
print("Training complete.")

In [ ]:
# In-sample predictions
preds = model.predict(X)
residuals = y - preds

res_mean = float(residuals.mean())
res_std  = float(residuals.std())

print(f"In-sample metrics")
print(f"  MAE  : {np.abs(residuals).mean():.4f}")
print(f"  RMSE : {np.sqrt((residuals**2).mean()):.4f}")
print(f"  Residual mean : {res_mean:.4f}")
print(f"  Residual std  : {res_std:.4f}")
print(f"  Anomaly threshold (±{ANOMALY_Z_THRESHOLD}σ): ±{ANOMALY_Z_THRESHOLD * res_std:.4f} units")

---
## 5. Model Inspection

### 5a. Feature Importances

In [ ]:
importances = model.feature_importances_

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(feature_names, importances, color=["steelblue"] * LAG_WINDOW + ["darkorange", "darkorange"])
ax.set_xlabel("Feature importance (gain)")
ax.set_title(f"{DEMO_SIGNAL} — XGBoost feature importances")
ax.invert_yaxis()

for bar, val in zip(bars, importances):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

print("\nExpected: lag_1 dominates (most recent value is the best predictor).")
print(f"lag_1 importance: {importances[0]:.3f}")

### 5b. Predicted vs Actual

In [ ]:
WINDOW = 200  # show last N buckets

ts_idx = resampled.index[LAG_WINDOW:]
ts_window = ts_idx[-WINDOW:]
y_window = y[-WINDOW:]
p_window = preds[-WINDOW:]

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Top: actual vs predicted
axes[0].plot(ts_window, y_window,  label="Actual",    linewidth=1.2, color="#22c55e")
axes[0].plot(ts_window, p_window,  label="Predicted", linewidth=1.0, color="#3b82f6",
             linestyle="--", alpha=0.85)
axes[0].set_title(f"{DEMO_SIGNAL} — last {WINDOW} buckets (25s each)")
axes[0].set_ylabel("value")
axes[0].legend()

# Bottom: residuals + threshold bands
res_window = y_window - p_window
axes[1].plot(ts_window, res_window, linewidth=0.8, color="gray", label="Residual")
axes[1].axhline(res_mean + ANOMALY_Z_THRESHOLD * res_std, color="red",   linestyle="--", linewidth=1, label=f"+{ANOMALY_Z_THRESHOLD}σ threshold")
axes[1].axhline(res_mean - ANOMALY_Z_THRESHOLD * res_std, color="red",   linestyle="--", linewidth=1, label=f"-{ANOMALY_Z_THRESHOLD}σ threshold")
axes[1].axhline(0, color="black", linewidth=0.5, alpha=0.5)
axes[1].set_ylabel("residual (actual − predicted)")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 6. Residual Analysis

Verify that residuals are approximately normal and the z-score threshold makes sense.

In [ ]:
z_scores = (residuals - res_mean) / res_std

print("Z-score distribution")
print(f"  Mean   : {z_scores.mean():.4f}  (should be ≈ 0)")
print(f"  Std    : {z_scores.std():.4f}   (should be ≈ 1)")
print(f"  Min    : {z_scores.min():.3f}")
print(f"  Max    : {z_scores.max():.3f}")

in_sample_anomalies = np.sum(np.abs(z_scores) > ANOMALY_Z_THRESHOLD)
pct = 100 * in_sample_anomalies / len(z_scores)
print(f"\nIn-sample buckets with |z| > {ANOMALY_Z_THRESHOLD}: {in_sample_anomalies} / {len(z_scores)} ({pct:.2f}%)")
print("For a normal distribution, expect ≈ 0.27% at 3σ.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(z_scores, bins=60, color="steelblue", alpha=0.7, edgecolor="white", linewidth=0.3)
axes[0].axvline( ANOMALY_Z_THRESHOLD, color="red", linestyle="--", label=f"+{ANOMALY_Z_THRESHOLD}σ")
axes[0].axvline(-ANOMALY_Z_THRESHOLD, color="red", linestyle="--", label=f"-{ANOMALY_Z_THRESHOLD}σ")
axes[0].set_xlabel("z-score")
axes[0].set_ylabel("count")
axes[0].set_title("In-sample z-score distribution")
axes[0].legend()

# QQ-style: sorted z-scores vs expected normal quantiles
from scipy import stats as scipy_stats
(osm, osr), (slope, intercept, r) = scipy_stats.probplot(z_scores, dist="norm")
axes[1].plot(osm, osr, "o", markersize=2, alpha=0.5, color="steelblue")
axes[1].plot(osm, slope * np.array(osm) + intercept, "r-", linewidth=1.5, label=f"Normal fit (R={r:.3f})")
axes[1].set_xlabel("theoretical quantiles")
axes[1].set_ylabel("sample quantiles")
axes[1].set_title("Q-Q plot of residuals")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 7. Anomaly Injection Tests

We take a clean window of real data, inject a synthetic attack, and verify:
- Normal buckets → z-score < 3 → no anomaly flagged
- Attacked buckets → z-score > 3 → anomaly flagged
- After 10 consecutive anomalies → `sustained = True`
- Imputation: model keeps predicting as if attack didn't happen (history uses predicted, not actual)

In [ ]:
def simulate_inference(series_slice: pd.Series, attack_start: int, attack_end: int, attack_delta: float):
    """
    Step through a series slice with the same logic as state.process_bucket().

    attack_start / attack_end: bucket indices where the injected attack is active.
    attack_delta: value added to the real reading during the attack window.

    Returns a DataFrame with per-bucket results.
    """
    history = deque(maxlen=LAG_WINDOW)
    consecutive_anomalies = 0

    rows = []
    for i, (ts, real_val) in enumerate(series_slice.items()):
        ts = pd.Timestamp(ts)
        # Inject attack
        actual = float(real_val) + (attack_delta if attack_start <= i < attack_end else 0.0)
        attacked = attack_start <= i < attack_end

        if len(history) < LAG_WINDOW:
            history.append(actual)
            rows.append({"i": i, "ts": ts, "actual": actual, "predicted": None,
                         "z_score": None, "is_anomaly": False, "attacked": attacked,
                         "consecutive": 0, "sustained": False, "status": "warming_up"})
            continue

        lags = list(reversed(history))
        features = np.array([[*lags, ts.hour, ts.minute]], dtype=np.float32)
        predicted = float(model.predict(features)[0])
        residual = actual - predicted
        z_score = (residual - res_mean) / res_std if res_std > 1e-9 else 0.0

        is_anomaly = abs(z_score) > ANOMALY_Z_THRESHOLD
        if is_anomaly:
            history.append(predicted)   # impute — don't let attack poison future preds
            consecutive_anomalies += 1
        else:
            history.append(actual)
            consecutive_anomalies = 0

        sustained = consecutive_anomalies >= SUSTAINED_ANOMALY_THRESHOLD
        rows.append({"i": i, "ts": ts, "actual": actual, "predicted": predicted,
                     "z_score": z_score, "is_anomaly": is_anomaly, "attacked": attacked,
                     "consecutive": consecutive_anomalies, "sustained": sustained,
                     "status": ("anomaly" if is_anomaly else "normal")})

    return pd.DataFrame(rows).set_index("i")

print("simulate_inference() defined.")

### Test 1: Voltage spike attack (+20V for 15 buckets)

In [ ]:
test_slice = resampled.iloc[-120:]   # last 120 buckets (10 minutes)
ATTACK_START = 40
ATTACK_END   = 55
ATTACK_DELTA = 20.0  # +20V spike

results = simulate_inference(test_slice, ATTACK_START, ATTACK_END, ATTACK_DELTA)

normal_rows   = results[~results["attacked"] & (results["status"] == "normal")]
anomaly_rows  = results[results["is_anomaly"]]
attacked_rows = results[results["attacked"]]

print("── Test 1: +20V spike ──")
print(f"Total buckets     : {len(results)}")
print(f"Attack window     : buckets {ATTACK_START}–{ATTACK_END-1} ({ATTACK_END - ATTACK_START} buckets)")
print(f"Anomalies flagged : {results['is_anomaly'].sum()}")
print(f"Detection rate    : {results.loc[results['attacked'], 'is_anomaly'].mean():.0%} of attacked buckets caught")
print(f"False positive rate: {results.loc[~results['attacked'] & results['z_score'].notna(), 'is_anomaly'].mean():.2%}")
print(f"Sustained alert   : {results['sustained'].any()}  (triggered after {SUSTAINED_ANOMALY_THRESHOLD} consecutive)")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# --- actual vs predicted ---
ready = results[results["predicted"].notna()]
axes[0].plot(ready.index, ready["actual"],    label="Actual",    color="#22c55e", linewidth=1.2)
axes[0].plot(ready.index, ready["predicted"], label="Predicted", color="#3b82f6", linewidth=1.0, linestyle="--")
# shade attack window
axes[0].axvspan(ATTACK_START, ATTACK_END - 1, alpha=0.15, color="red", label="Attack window")
# mark detected anomalies
det = results[results["is_anomaly"]]
axes[0].scatter(det.index, det["actual"], color="red", zorder=5, s=30, label="Detected anomaly")
axes[0].set_ylabel("value")
axes[0].set_title(f"Test 1: +{ATTACK_DELTA}V spike attack on {DEMO_SIGNAL}")
axes[0].legend()

# --- z-scores ---
ready_z = results[results["z_score"].notna()]
axes[1].plot(ready_z.index, ready_z["z_score"], color="gray", linewidth=0.9, label="Z-score")
axes[1].axhline( ANOMALY_Z_THRESHOLD, color="red",  linestyle="--", linewidth=1, label=f"+{ANOMALY_Z_THRESHOLD}σ")
axes[1].axhline(-ANOMALY_Z_THRESHOLD, color="red",  linestyle="--", linewidth=1)
axes[1].axhline(0, color="black", linewidth=0.4, alpha=0.5)
axes[1].axvspan(ATTACK_START, ATTACK_END - 1, alpha=0.15, color="red")
axes[1].set_ylabel("z-score")
axes[1].set_xlabel("bucket index")
axes[1].legend()

plt.tight_layout()
plt.show()

### Test 2: Sustained attack (30 buckets) → sustained flag

In [ ]:
results2 = simulate_inference(test_slice, ATTACK_START, ATTACK_START + 30, ATTACK_DELTA)

print("── Test 2: Sustained attack (30 buckets) ──")
print(f"Anomalies flagged : {results2['is_anomaly'].sum()}")
print(f"Sustained alert   : {results2['sustained'].any()}")

first_sustained = results2[results2["sustained"]].index[0] if results2["sustained"].any() else None
print(f"First sustained at bucket: {first_sustained}  (expected ≈ {ATTACK_START + SUSTAINED_ANOMALY_THRESHOLD - 1})")

### Test 3: Small noise (+1V) — should NOT trigger (below 3σ)

In [ ]:
small_delta = 0.5 * ANOMALY_Z_THRESHOLD * res_std   # half the threshold
results3 = simulate_inference(test_slice, ATTACK_START, ATTACK_END, small_delta)

print("── Test 3: Sub-threshold noise ──")
print(f"Attack delta     : +{small_delta:.4f}  ({0.5 * ANOMALY_Z_THRESHOLD:.1f}σ worth of noise)")
print(f"Anomalies flagged: {results3['is_anomaly'].sum()}  (expect 0 or very few)")
print(f"Max z-score in attack window: {results3.loc[ATTACK_START:ATTACK_END, 'z_score'].max():.2f}  (threshold={ANOMALY_Z_THRESHOLD})")

### Test 4: Imputation verification

When an anomaly is detected, the model uses the **predicted** value (not the actual attacked value) for future predictions.
This prevents the attack from corrupting the model's internal state.

In [ ]:
# Compare: what predictions look like after attack ends WITH vs WITHOUT imputation
def simulate_no_imputation(series_slice, attack_start, attack_end, attack_delta):
    """Same as simulate_inference but always appends actual (no imputation)."""
    history = deque(maxlen=LAG_WINDOW)
    rows = []
    for i, (ts, real_val) in enumerate(series_slice.items()):
        ts = pd.Timestamp(ts)
        actual = float(real_val) + (attack_delta if attack_start <= i < attack_end else 0.0)
        if len(history) < LAG_WINDOW:
            history.append(actual)
            rows.append({"i": i, "predicted": None})
            continue
        lags = list(reversed(history))
        features = np.array([[*lags, ts.hour, ts.minute]], dtype=np.float32)
        predicted = float(model.predict(features)[0])
        history.append(actual)   # no imputation
        rows.append({"i": i, "predicted": predicted})
    return pd.DataFrame(rows).set_index("i")

r_imputed     = simulate_inference(test_slice, ATTACK_START, ATTACK_END, ATTACK_DELTA)
r_no_impute   = simulate_no_imputation(test_slice, ATTACK_START, ATTACK_END, ATTACK_DELTA)

# Look at predictions in the window AFTER the attack ends
post_attack = slice(ATTACK_END, ATTACK_END + 20)
actual_clean = test_slice.iloc[ATTACK_END:ATTACK_END + 20].values

pred_imputed   = r_imputed.loc[ATTACK_END:ATTACK_END + 19, "predicted"].values
pred_no_impute = r_no_impute.loc[ATTACK_END:ATTACK_END + 19, "predicted"].values

mae_imputed   = np.abs(actual_clean - pred_imputed).mean()
mae_no_impute = np.abs(actual_clean - pred_no_impute).mean()

print("── Test 4: Post-attack prediction quality ──")
print(f"MAE after attack (WITH imputation)   : {mae_imputed:.4f}")
print(f"MAE after attack (WITHOUT imputation): {mae_no_impute:.4f}")
print(f"Imputation improvement               : {(mae_no_impute - mae_imputed) / mae_no_impute:.1%} better")

---
## 8. Inference Simulation — Step Through the Deque

Replicate exactly what `inference.py` does at runtime, bucket by bucket.

In [ ]:
print("Simulating live inference on last 30 buckets...\n")
print(f"{'bucket':>6} {'timestamp':>32} {'actual':>10} {'predicted':>10} {'z_score':>8} {'status'}")
print("-" * 85)

history = deque(maxlen=LAG_WINDOW)
live_slice = resampled.iloc[-30:]

for i, (ts_raw, val) in enumerate(live_slice.items()):
    ts = pd.Timestamp(ts_raw)
    actual = float(val)

    if len(history) < LAG_WINDOW:
        history.append(actual)
        print(f"{i:>6} {str(ts):>32} {actual:>10.4f} {'—':>10} {'—':>8}  warming up ({len(history)}/{LAG_WINDOW})")
        continue

    lags = list(reversed(history))
    features = np.array([[*lags, ts.hour, ts.minute]], dtype=np.float32)
    predicted = float(model.predict(features)[0])
    residual = actual - predicted
    z_score = (residual - res_mean) / res_std
    is_anomaly = abs(z_score) > ANOMALY_Z_THRESHOLD
    history.append(predicted if is_anomaly else actual)

    status = "ANOMALY" if is_anomaly else "normal"
    print(f"{i:>6} {str(ts):>32} {actual:>10.4f} {predicted:>10.4f} {z_score:>8.3f}  {status}")

---
## 9. All-Signals Test

Load every saved model, feed it the last 50 resampled buckets from its signal's CSV,
and verify it produces sensible predictions.

In [ ]:
import json

if not RESIDUAL_STATS_PATH.exists():
    print("residual_stats.json not found. Run train.py --all first.")
else:
    with open(RESIDUAL_STATS_PATH) as f:
        all_stats = json.load(f)

    print(f"{'Signal':<25} {'Buckets':>8} {'MAE':>8} {'Res std':>9} {'Threshold':>11} {'Model file'}")
    print("-" * 90)

    all_ok = True
    for sig_name, sig_cfg in SIGNALS.items():
        model_path = MODELS_DIR / f"{sig_name}.pkl"
        if not model_path.exists():
            print(f"{sig_name:<25}  MISSING pkl")
            all_ok = False
            continue
        if sig_name not in all_stats:
            print(f"{sig_name:<25}  MISSING stats")
            all_ok = False
            continue

        with open(model_path, "rb") as f:
            m = pickle.load(f)

        raw_s = load_sensor_logs(sig_cfg["sensor"], sig_cfg["field"], LOG_DIR)
        rs = resample_to_5s(raw_s)

        if len(rs) < LAG_WINDOW + 5:
            print(f"{sig_name:<25}  not enough data to test")
            all_ok = False
            continue

        test_s = rs.iloc[-(50 + LAG_WINDOW):]
        X_t, y_t = build_features(test_s, LAG_WINDOW)
        p_t = m.predict(X_t)
        mae = float(np.abs(y_t - p_t).mean())
        stat = all_stats[sig_name]
        threshold = ANOMALY_Z_THRESHOLD * stat["std"]

        print(f"{sig_name:<25} {len(X_t):>8} {mae:>8.4f} {stat['std']:>9.4f} {threshold:>11.4f}  {model_path.name}")

    print()
    print("All models OK." if all_ok else "Some models missing — run train.py --all")

---
## 10. Single-Bucket Prediction — Manual Input Test

Feed the model exactly one feature vector you construct by hand and see what it predicts.
Useful for sanity-checking a specific scenario.

In [ ]:
# ── Edit these values to test whatever you want ──
manual_lags = [230.1, 230.3, 230.0, 229.9, 230.2]   # [v(t-1), v(t-2), ..., v(t-5)]
manual_hour   = 14
manual_minute = 32
# ─────────────────────────────────────────────────

feat = np.array([[*manual_lags, manual_hour, manual_minute]], dtype=np.float32)
pred_val = float(model.predict(feat)[0])

print(f"Input lags (t-1 → t-5): {manual_lags}")
print(f"Time of day           : {manual_hour:02d}:{manual_minute:02d}")
print(f"Predicted value       : {pred_val:.4f}")
print()
print("Now inject an anomaly — add 20V to the incoming actual:")
injected_actual = pred_val + 20.0
residual = injected_actual - pred_val
z = (residual - res_mean) / res_std
print(f"  actual         = {injected_actual:.4f}")
print(f"  predicted      = {pred_val:.4f}")
print(f"  residual       = {residual:.4f}")
print(f"  z-score        = {z:.3f}")
print(f"  is_anomaly     = {abs(z) > ANOMALY_Z_THRESHOLD}  (threshold={ANOMALY_Z_THRESHOLD})")